In [5]:
# packages and working directory  
import scipy 
import sklearn
import econml 
import arch
import os 
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib as plt
import statsmodels.api as sm 
from statsmodels.discrete.discrete_model import Probit
from statsmodels.iolib.summary2 import summary_col
import statsmodels.formula.api as smf 
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt 
from scipy import stats
from scipy.stats import ttest_ind
from scipy.optimize import approx_fprime

from torch_choice.data import ChoiceDataset
from torch_choice.model import ConditionalLogitModel
# Consolidate changing directory and CPI dictionary since these don't change throughout the script
new_directory = r'C:\Users\hisham\Spain\2021 datasets'
os.chdir(new_directory)



In [6]:
df = pd.read_csv('4choicestalong.csv')

# Define the list of scenarios
scenarios = [0, 1, 2, 3]

# Create the 'labor' column based on the 'scenario' column
df['labor'] = df.apply(lambda row: row[f'lhw_{int(row["scenario"])}'], axis=1)

df['log_y'] = np.log(df['ils_udb_yds'])
df['log_l'] = np.log(80 - df['labor'])
df['log2_y'] = 0.5* df['log_y']**2 
df['log2_l'] = 0.5* df['log_l']**2
df['log_y_l'] = df['log_y'] * df['log_l']

In [7]:
df.to_stata('4choices.dta')


In [14]:

def utility_extended(params, row):
    alpha, beta, gamma_yy, gamma_ll, gamma_yl = params
    log_y = row['log_y']
    log_l = row['log_l']
    utility_value = alpha * log_y + beta * log_l + \
                    0.5 * gamma_yy * (log_y**2) + 0.5 * gamma_ll * (log_l**2) + \
                    gamma_yl * log_y * log_l
    mu_y = (alpha + gamma_yy * log_y + gamma_yl* log_l)
    mu_l = (beta + gamma_ll * log_l + gamma_yl * log_y )
    return utility_value, mu_y, mu_l

# Calculate probabilities for each person based on utility values
def calculate_probabilities_corrected(group):
    utilities = group['utility'].values
    exp_utilities = np.exp(utilities - np.max(utilities))
    probabilities = exp_utilities / np.sum(exp_utilities)
    return pd.Series(probabilities, index=group.index)




In [15]:
params = [11.71094, 38.96736, -1.230868, -9.597563, -0.5062818]

In [16]:
df[['utility', 'mu_y', 'mu_l']] = df.apply(lambda row: utility_extended(params, row), axis=1, result_type="expand")
# Applying the function correctly
df['probability'] = df.groupby('idperson').apply(lambda group: calculate_probabilities_corrected(group)).reset_index(level=0, drop=True)


# Calculate the max probability for each group
max_probs = df.groupby('idperson')['probability'].transform('max')

# Assign predicted choice based on whether the probability equals the max probability within its group
df['predicted_choice'] = (df['probability'] == max_probs).astype(int)

# Count of people with negative marginal utility of leisure at chosen points
negative_MU_l_chosen = df[(df['choice_made'] == 1) & (df['mu_l'] < 0)]['idperson'].nunique()

# Count of people with negative marginal utility of leisure at predicted points
negative_MU_l_predicted = df[(df['predicted_choice'] == 1) & (df['mu_l'] < 0)]['idperson'].nunique()

# Count of people with negative marginal utility of income at chosen points
negative_MU_y_chosen = df[(df['choice_made'] == 1) & (df['mu_y'] < 0)]['idperson'].nunique()

# Count of people with negative marginal utility of income at predicted points
negative_MU_y_predicted = df[(df['predicted_choice'] == 1) & (df['mu_y'] < 0)]['idperson'].nunique()

print("Negative MU_l at chosen points:", negative_MU_l_chosen)
print("Negative MU_l at predicted points:", negative_MU_l_predicted)
print("Negative MU_y at chosen points:", negative_MU_y_chosen)
print("Negative MU_y at predicted points:", negative_MU_y_predicted)

Negative MU_l at chosen points: 3237
Negative MU_l at predicted points: 3456
Negative MU_y at chosen points: 486
Negative MU_y at predicted points: 561


In [18]:
df['log_l'].describe()

count    17180.000000
mean         3.792385
std          0.596205
min          1.609438
25%          3.457799
50%          3.979638
75%          4.372593
max          4.382027
Name: log_l, dtype: float64

In [47]:
# Define your parameter values
beta_log_y = 11.71094
beta_log2_y = -1.230868
beta_log_y_l = -0.5062818

# Assuming an example average log(l), you should replace it with the actual mean from your data if needed
average_log_l = np.log(70 - np.mean(df['labor']))  # example calculation

# Calculate the income level at which the marginal utility of income becomes zero
log_y = (-beta_log_y - beta_log_y_l * average_log_l) / (beta_log2_y)
ils_udb_yds_level = np.exp(log_y)

print(f"At ils_udb_yds = {ils_udb_yds_level:.2f}, the marginal utility of income becomes zero.")


At ils_udb_yds = 2929.24, the marginal utility of income becomes zero.


In [48]:
df1 = df[df['predicted_choice'] == 1 ]
df2 = df[df['choice_made'] == 1 ]

In [49]:
# Define your parameter values
beta_log_y = 11.71094
beta_log2_y = -1.230868
beta_log_y_l = -0.5062818
beta_log_l = 38.96736
beta_log2_l = -9.597563
# Assuming an example average log(l), you should replace it with the actual mean from your data if needed
average_log_l = np.log(80 - np.mean(df1['labor']))  # example calculation

# Calculate the income level at which the marginal utility of income becomes zero
log_y = (-beta_log_y - beta_log_y_l * average_log_l) / ( beta_log2_y)
ils_udb_yds_level = np.exp(log_y)



# Assuming an example average log(y), you should replace it with the actual mean or a specific income level from your data
average_log_y = np.log(np.mean(df['ils_udb_yds']))  # example calculation

# Calculate the leisure level at which the marginal utility of leisure becomes zero
log_l = (-beta_log_l - beta_log_y_l * average_log_y) / ( beta_log2_l)
leisure_level = np.exp(log_l)

print(f"At leisure time = {leisure_level:.2f} hours, the marginal utility of leisure becomes zero.")


print(f"At ils_udb_yds = {ils_udb_yds_level:.2f}, the marginal utility of income becomes zero.")


At leisure time = 39.29 hours, the marginal utility of leisure becomes zero.
At ils_udb_yds = 2934.39, the marginal utility of income becomes zero.


In [51]:
df1['leisure'] = 75 - df['labor']
df2['leisure'] = 75- df['labor']


C:\Users\hisham\AppData\Local\Temp\ipykernel_20512\3689305431.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['leisure'] = 75 - df['labor']
C:\Users\hisham\AppData\Local\Temp\ipykernel_20512\3689305431.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['leisure'] = 75- df['labor']


In [52]:
df1[['ils_udb_yds', 'leisure' ]].describe()



,ils_udb_yds,leisure
count,4295.000000,4295.000000
mean,1953.330021,36.264959
std,1082.551090,3.577750
min,316.310000,22.000000
25%,1256.060000,35.000000
50%,1728.050000,35.000000
75%,2391.555000,38.000000
max,12669.210000,48.000000


In [53]:
# Define your parameter values
beta_log_y = 11.71094
beta_log2_y = -1.230868
beta_log_y_l = -0.5062818
beta_log_l = 38.96736
beta_log2_l = -9.597563
# Assuming an example average log(l), you should replace it with the actual mean from your data if needed
average_log_l = np.log(80 - np.mean(df2['labor']))  # example calculation

# Calculate the income level at which the marginal utility of income becomes zero
log_y = (-beta_log_y - beta_log_y_l * average_log_l) / ( beta_log2_y)
ils_udb_yds_level = np.exp(log_y)



# Assuming an example average log(y), you should replace it with the actual mean or a specific income level from your data
average_log_y = np.log(np.mean(df2['ils_udb_yds']))  # example calculation

# Calculate the leisure level at which the marginal utility of leisure becomes zero
log_l = (-beta_log_l - beta_log_y_l * average_log_y) / ( beta_log2_l)
leisure_level = np.exp(log_l)

print(f"At leisure time = {leisure_level:.2f} hours, the marginal utility of leisure becomes zero.")


print(f"At ils_udb_yds = {ils_udb_yds_level:.2f}, the marginal utility of income becomes zero.")


At leisure time = 38.96 hours, the marginal utility of leisure becomes zero.
At ils_udb_yds = 2872.52, the marginal utility of income becomes zero.


In [54]:
negulcc = df2[df2['ils_udb_yds'] > 2872.52]
negullc = df2[df2['leisure'] > 38.96]
negulcp = df1[df1['ils_udb_yds'] > 2934.39]
negullp = df1[df1['leisure'] > 39.29]

print(negulcc.shape , negulcp.shape , negullc.shape,negullp.shape )

(591, 361) (555, 361) (1252, 361) (814, 361)


In [55]:
df2[['ils_udb_yds', 'leisure' ]].describe()

,ils_udb_yds,leisure
count,4295.000000,4295.000000
mean,1872.909129,38.459139
std,1086.593300,12.511425
min,330.000000,0.000000
25%,1158.900000,35.000000
50%,1637.830000,35.000000
75%,2380.995000,40.000000
max,13246.500000,75.000000


In [59]:
# Define your parameter values
beta_log_y = 10.31376
beta_log2_y = -1.006
beta_log_y_l = -.4884251
beta_log_l = 36.507
beta_log2_l = -8.844
# Assuming an example average log(l), you should replace it with the actual mean from your data if needed
average_log_l = np.log(80 - np.mean(df1['labor']))  # example calculation

# Calculate the income level at which the marginal utility of income becomes zero
log_y = (-beta_log_y - beta_log_y_l * average_log_l) / ( beta_log2_y)
ils_udb_yds_level = np.exp(log_y)



# Assuming an example average log(y), you should replace it with the actual mean or a specific income level from your data
average_log_y = np.log(np.mean(df1['ils_udb_yds']))  # example calculation

# Calculate the leisure level at which the marginal utility of leisure becomes zero
log_l = (-beta_log_l - beta_log_y_l * average_log_y) / ( beta_log2_l)
leisure_level = np.exp(log_l)

print(f"At leisure time = {leisure_level:.2f} hours, the marginal utility of leisure becomes zero.")


print(f"At ils_udb_yds = {ils_udb_yds_level:.2f}, the marginal utility of income becomes zero.")


At leisure time = 40.83 hours, the marginal utility of leisure becomes zero.
At ils_udb_yds = 4657.04, the marginal utility of income becomes zero.


In [65]:
negulcc = df2[df2['ils_udb_yds'] > 4657.04]
negullc = df2[df2['leisure'] > 40.83]
negulcp = df1[df1['ils_udb_yds'] > 4657.04]
negullp = df1[df1['leisure'] > 40.83]

len(negulcc) 
len(negulcp)
len(negullc)
len(negullp)

425

In [68]:
len(negullp)

425

In [72]:
# Assuming `df_filtered1` has columns: 'idperson', 'scenario', 'choice_made', 'predicted_choice'
# And 'scenario' column contains the scenario labels ('h0', 'h1', 'h2', 'h3')

# Marking actual choice scenarios (assuming 'choice_made' directly indicates this)
df['actual_choice'] = df.apply(lambda x: x['scenario'] if x['choice_made'] == 1 else None, axis=1)
df['actual_choice'] = df.groupby('idperson')['actual_choice'].ffill().bfill()

# Identifying the predicted choice scenario
df['predicted_choice_scenario'] = df.apply(lambda x: x['scenario'] if x['predicted_choice'] == 1 else None, axis=1)
df['predicted_choice_scenario'] = df.groupby('idperson')['predicted_choice_scenario'].ffill().bfill()

# Ensure there's no individual without a predicted scenario
assert df['predicted_choice_scenario'].isnull().sum() == 0, "Some individuals don't have a predicted scenario."

# First, ensure 'predicted_choice' is correctly assigned in 'df_filtered1'
# This step assumes you have a method to assign 'predicted_choice' correctly based on your model's output

# For simplicity, let's rebuild 'predicted_choice_scenario' based on the highest utility or probability
# Assuming 'utility' or 'probability' column exists in 'df_filtered1' indicating model output

# Step 1: Assign predicted choice scenario based on the highest utility or probability
# This method assumes 'utility' column exists; adjust accordingly if using probabilities
df['predicted_choice'] = df.groupby('idperson')['utility'].transform(lambda x: x == x.max())

# Now, each individual should have exactly one 'predicted_choice' flagged
# Let's try assigning 'predicted_choice_scenario' again
df['predicted_choice_scenario'] = df.apply(
    lambda x: x['scenario'] if x['predicted_choice'] else None, axis=1
)

# Fill missing 'predicted_choice_scenario' within each group
df['predicted_choice_scenario'] = df.groupby('idperson')['predicted_choice_scenario'].ffill().bfill()

# Verify the assignment
missing_predictions_post_fix = df[df['choice_made'] == 1]['predicted_choice_scenario'].isnull().sum()
if missing_predictions_post_fix > 0:
    raise AssertionError(f"Post-fix, still missing predictions for {missing_predictions_post_fix} individuals.")

# Assuming no error is raised, proceed with creating the comparison table
# This time, ensure to use the adjusted dataset where predicted scenarios are guaranteed to be assigned
actual_choices_df = df[df['choice_made'] == 1].copy()
comparison_table = pd.crosstab(
    actual_choices_df['actual_choice'],
    actual_choices_df['predicted_choice_scenario'],
    margins=True,
    margins_name='Total'
)

print(comparison_table)


predicted_choice_scenario   2.0  3.0  Total
actual_choice                              
0.0                         199    0    199
1.0                         479    0    479
2.0                        3181    0   3181
3.0                         416   20    436
Total                      4275   20   4295
